# 🚀 Comprehensive Cloud Processing Tool (LZMA + Bsdiff + Aria2)

Welcome! This tool allows you to download files, create patches, and compress them at high speeds using Google Colab's cloud resources.

### 🛠️ How to Use:
1. Press the play button (▶️) on the left side of the code cell below, or press `Ctrl+Enter` while inside the cell.
2. Wait a few seconds for the system to check requirements and install necessary tools automatically.
3. A graphical user interface (Gradio) will appear directly below the cell.
4. Use the interface to enter file links and execute the required operations.
5. You can open the interface in a new window via the Public URL that will appear later.

In [ ]:
import os
import sys
import shutil
import subprocess
import hashlib
import tempfile
import zlib

# ==========================================
# 1. Smart Installation System
# ==========================================
def setup_environment():
    print("🔍 Checking system...")
    needs_apt = not (shutil.which("bsdiff") and shutil.which("aria2c") and shutil.which("xz"))
    if needs_apt:
        print("⚙️ Installing system tools (bsdiff, aria2, xz-utils)...")
        subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(["apt-get", "install", "-y", "-qq", "bsdiff", "aria2", "xz-utils"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        import gradio
    except ImportError:
        print("⚙️ Installing GUI (Gradio)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ System is fully ready.")

setup_environment()
import gradio as gr

# ==========================================
# 2. Core Processing Functions
# ==========================================
def download_file(url, output_path):
    output_dir = os.path.dirname(output_path)
    if not output_dir:
        output_dir = "."
    output_name = os.path.basename(output_path)
    command = ["aria2c", "-x", "16", "-s", "16", "--seed-time=0", "-d", output_dir, "-o", output_name, url]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise Exception(f"Failed to download from link: {url}\nReason: {result.stderr}")

def calculate_hashes(filepath):
    md5 = hashlib.md5()
    sha1 = hashlib.sha1()
    sha256 = hashlib.sha256()
    sha384 = hashlib.sha384()
    sha512 = hashlib.sha512()
    crc32_val = 0

    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b""):
            md5.update(chunk)
            sha1.update(chunk)
            sha256.update(chunk)
            sha384.update(chunk)
            sha512.update(chunk)
            crc32_val = zlib.crc32(chunk, crc32_val)

    return {
        "CRC32": "%08X" % (crc32_val & 0xFFFFFFFF),
        "MD5": md5.hexdigest().upper(),
        "SHA1": sha1.hexdigest().upper(),
        "SHA256": sha256.hexdigest().upper(),
        "SHA384": sha384.hexdigest().upper(),
        "SHA512": sha512.hexdigest().upper()
    }

def create_patch(original, modified, patch_name):
    command = ["bsdiff", original, modified, patch_name]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise Exception(f"Patch creation failed!\nReason: {result.stderr}")
    return patch_name

def compress_file_lzma(file_name):
    command = ["xz", "-z", "-9", "-e", "-T0", file_name]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise Exception(f"LZMA compression failed!\nReason: {result.stderr}")
    return f"{file_name}.xz"

def process(operation_mode, original_url, modified_url):
    if not original_url.strip():
        raise gr.Error("Please enter at least the first link!")

    temp_dir = tempfile.mkdtemp()
    orig_file = os.path.join(temp_dir, "downloaded_file.bin")
    mod_file = os.path.join(temp_dir, "modified_file.bin")
    patch_file = os.path.join(temp_dir, "output.patch")

    try:
        if operation_mode == "Download file and compress directly":
            gr.Info("📥 Starting file download...")
            download_file(original_url, orig_file)
            
            gr.Info("🔍 Extracting file hashes...")
            hashes = calculate_hashes(orig_file)
            
            msg = f"### ✅ Downloaded and compressed file information:\n* **MD5:** `{hashes['MD5']}`\n* **SHA256:** `{hashes['SHA256']}`\n* **CRC32:** `{hashes['CRC32']}`"
            
            gr.Info("🗜️ Compressing file with maximum settings (LZMA)...")
            final_file = compress_file_lzma(orig_file)
            return final_file, msg

        else:
            if not modified_url.strip():
                raise gr.Error("In patch creation mode, you must enter the modified file link!")
                
            gr.Info("📥 Starting original file download...")
            download_file(original_url, orig_file)
            
            gr.Info("🔍 Calculating original file hashes...")
            hashes = calculate_hashes(orig_file)
            
            msg = f"### ⚠️ Warning: These values must match your original file for the patch to succeed!\n* **MD5:** `{hashes['MD5']}`\n* **SHA256:** `{hashes['SHA256']}`\n* **CRC32:** `{hashes['CRC32']}`"
            
            gr.Info("📥 Starting modified file download...")
            download_file(modified_url, mod_file)
            
            gr.Info("⚙️ Creating patch (this may take some time)...")
            create_patch(orig_file, mod_file, patch_file)
            
            gr.Info("🗜️ Compressing patch with maximum settings (LZMA)...")
            final_file = compress_file_lzma(patch_file)
            
            return final_file, msg

    except Exception as e:
        raise gr.Error(str(e))

with gr.Blocks(theme=gr.themes.Soft()) as interface:
    gr.Markdown("# 🚀 Comprehensive Cloud Processing Tool (LZMA + Bsdiff + Aria2)")
    
    mode_selector = gr.Radio(
        choices=["Create patch between two files and compress", "Download file and compress directly"],
        value="Create patch between two files and compress",
        label="🛠️ Choose Operation Type",
        interactive=True
    )
    
    with gr.Row():
        with gr.Column():
            orig_input = gr.Textbox(label="🔗 Original File Link", placeholder="https://... or magnet:?xt=...")
            mod_input = gr.Textbox(label="🔗 Modified File Link", placeholder="https://... or magnet:?xt=...")
            run_btn = gr.Button("⚡ Start Process", variant="primary")
            
        with gr.Column():
            output_file = gr.File(label="📥 Download Final File (.xz)")
            output_hashes = gr.Markdown(label="File Info (Hash)")

    def update_ui(mode):
        if mode == "Download file and compress directly":
            return gr.update(label="🔗 Link to download and compress"), gr.update(visible=False)
        else:
            return gr.update(label="🔗 Original File Link"), gr.update(visible=True)

    mode_selector.change(
        fn=update_ui,
        inputs=mode_selector,
        outputs=[orig_input, mod_input]
    )

    run_btn.click(
        fn=process,
        inputs=[mode_selector, orig_input, mod_input],
        outputs=[output_file, output_hashes]
    )

interface.launch(debug=True, share=True)